In [1]:
import cv2
import numpy as np
from scipy.signal import butter, filtfilt
import time
import mediapipe as mp

# Bandpass filter
def butter_bandpass_filter(data, lowcut=0.75, highcut=3.0, fs=30, order=5):
    nyq = 0.5 * fs
    low, high = lowcut / nyq, highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

# Initialize Mediapipe Face Detection
mp_face = mp.solutions.face_detection
face_detection = mp_face.FaceDetection(model_selection=0, min_detection_confidence=0.5)

cap = cv2.VideoCapture(0)

buffer_size = 300
green_values = []
timestamps = []

bpm = None
last_bpm_time = time.time()
bpm_interval = 3  # seconds between updates
bpm_history = []

print("Starting webcam feed...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_detection.process(rgb_frame)

    if results.detections:
        for detection in results.detections:
            bbox = detection.location_data.relative_bounding_box
            h, w, _ = frame.shape
            x, y, w_box, h_box = int(bbox.xmin * w), int(bbox.ymin * h), int(bbox.width * w), int(bbox.height * h)
            forehead = frame[y:y+int(h_box*0.2), x+int(w_box*0.3):x+int(w_box*0.7)]

            if forehead.size > 0:
                green = np.mean(forehead[:, :, 1])
                green_values.append(green)
                timestamps.append(time.time())

                if len(green_values) > buffer_size:
                    green_values.pop(0)
                    timestamps.pop(0)

                cv2.rectangle(frame, (x+int(w_box*0.3), y), (x+int(w_box*0.7), y+int(h_box*0.2)), (0, 255, 0), 2)

                # Update BPM estimate only every 3 seconds
                if len(green_values) == buffer_size and (time.time() - last_bpm_time) > bpm_interval:
                    signal = np.array(green_values)
                    filtered = butter_bandpass_filter(signal)
                    freqs = np.fft.fftfreq(len(filtered), d=(timestamps[1] - timestamps[0]))
                    fft_values = np.abs(np.fft.fft(filtered))
                    idx = np.where((freqs >= 0.75) & (freqs <= 3.0))
                    
                    peak_freq = freqs[idx][np.argmax(fft_values[idx])]
                    bpm = peak_freq * 60.0
                    last_bpm_time = time.time()

                    # Add to BPM history for smoothing
                    bpm_history.append(bpm)
                    if len(bpm_history) > 5:
                        bpm_history.pop(0)

    # Draw BPM
    if bpm_history:
        smoothed_bpm = int(np.mean(bpm_history))
        cv2.putText(frame, f"Estimated BPM: {smoothed_bpm}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Heart Rate Estimation", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Starting webcam feed...


ZeroDivisionError: float division by zero